# Setup

## Libraries

In [1]:
import os
from pathlib import Path

In [2]:
import pandas as pd

In [3]:
from datasets import load_dataset

/home/hong-mai/Desktop/HONGMAI/Github/recommender-systems-bootcamp/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [35]:
import torch
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset
from transformers import AutoTokenizer

In [19]:
import torch.nn as nn

## Environment

In [5]:
PROJECT_ROOT = Path(os.getcwd()).parent

MOVIES_SMALL_DIR = PROJECT_ROOT / "data/ml-latest-small"

# Test

## Data

### IMDB

In [ ]:
dataset = load_dataset("imdb")

In [3]:
print(dataset)
print(dataset["train"][0])

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})
{'text': 'I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and

In [6]:
dataset['train']['text'][3]

"This film was probably inspired by Godard's Masculin, féminin and I urge you to see that film instead.<br /><br />The film has two strong elements and those are, (1) the realistic acting (2) the impressive, undeservedly good, photo. Apart from that, what strikes me most is the endless stream of silliness. Lena Nyman has to be most annoying actress in the world. She acts so stupid and with all the nudity in this film,...it's unattractive. Comparing to Godard's film, intellectuality has been replaced with stupidity. Without going too far on this subject, I would say that follows from the difference in ideals between the French and the Swedish society.<br /><br />A movie of its time, and place. 2/10."

In [8]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

In [ ]:
text = "Transformers are powerful models."

encoded = tokenizer(
    text,
    padding="max_length",
    truncation=True,
    max_length=128
)

print(encoded)

['transformers', 'are', 'powerful', 'models', '.']


In [ ]:
text_list = [
    "Transformers are powerful models.",
    "Supersuper nova lova."
]

encodings = tokenizer(
    text_list,  # convert to list
    truncation=True,
    stride=32,
    return_overflowing_tokens=True,
    padding="max_length",
    max_length=128
)

encodings
# overflow_to_sample_mapping tells which original sample each produced chunk came from

{'input_ids': [[101, 19081, 2024, 3928, 4275, 1012, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [101, 3565, 6342, 4842, 6846, 8840, 3567, 1012, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]], 'token_type_ids': [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 

In [12]:
def tokenize_function(sequence):
    return tokenizer(
        sequence["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

In [13]:
tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True
)

Map: 100%|██████████| 50000/50000 [00:05<00:00, 9488.20 examples/s] 


### MovieLens

In [9]:
ratings_df = pd.read_csv(MOVIES_SMALL_DIR / "ratings.csv")
item_sequence = (
    ratings_df
    .sort_values(['userId','timestamp'])
    .groupby('userId')['movieId']
    .agg(list)
)
sentences = [[str(i) for i in seq] for seq in item_sequence]

In [14]:
len(sentences[0])

232

## Model

In [ ]:
class Encoder(nn.Module):
    '''
        Encode the input sequence S_u into embeddings.
        The encoder is shared between the augmenter and the recommender.
        Embedding matrix: E in R^{|I|*e}
            + e is the dimension of the embedding vector
            + I is the number of candidate items
        Steps:
            + Map
                + i_t -> e_t
                    - where i_t is an item in the input sequence S_u
                    - e_t in R^{e}
                + i_t -> p_t
                    - where p_t is the position embedding at the t-th position
            + Inject position information
                + h_t^0 = e_t + p_t
                    - where h0 is the initial hidden representation of an item
            + Update
                + H^l_e = Trm(H_e^{l-1})
                    - where H^l is the hidden representation matrix at layer l
                    - Trm is a transfomer block
        The last layer of H_e^L can be used as input for the augmenter and recommender.
    '''

    def __init__(self, n_items, seq_len, n_embed):
        
        super().__init__()

        self.token_embedder = nn.Embedding(n_items, n_embed)
        self.pos_embedder = nn.Embedding(seq_len, n_embed)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=n_embed,
            nhead=8
        )
        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=5
        )

    def forward(self, seq):

        token = self.token_embedder(seq)

        positions = torch.arange(seq.size(1))
        positions = positions.unsqueeze(0)
        pos = self.pos_embedder(positions)

        x = token + pos

        x = self.transformer(x)

        return x
    
class SSLAugmenter(nn.Module):

    def __init__(self):
        pass

    def choose_operation(self, seq):
        pass

    def delete(self):
        pass

    def keep(self):
        pass

    def skip(self):
        pass

class RandomAugmenter(nn.Module):
    pass

class Augmenter:
    '''
        Modifies the input sequence.
        Determine the operation to perform on each position in S_u: keep, delete or insert.
        Operations:
            + insert:
                - reverse generator: calculates the probability of each candidate item being inserted
                - insert subsequence in reverse
        Training:
            + Randomly modify S_u
            + Require the augmenter to restore it through a self-supervised training task.
        Inference:
            + Input: N raw sequences in a batch
            + Actions:
                - Create first S_u^{aug1} using SSL Augmenter
                - Create second S_u^{aug1} using random methods
    '''
    pass

class ContrastiveLearner:
    '''
        Positive pair: 2 augmented sequences derived from the same raw sequence.
        Negative pairs: 2N - 2 other sequences not derived from the current raw sequence.
        Triplet contrastive loss: L_{tri}
            + Objective: Reduce distance between raw sequence and SSL-augmented sequence
    '''


    pass

class Recommender:
    '''
        Predict user's next interaction.
    '''
    pass 

In [42]:
seq_len = 50
n_embed = 32
n_items = ml_dataset.num_items

encoder = Encoder(n_items=n_items, seq_len=seq_len, n_embed=n_embed)

/tmp/ipykernel_84639/1175440500.py:36: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  self.transformer = nn.TransformerEncoder(


In [43]:
batch = next(iter(loader))

print(batch["input_ids"].shape)
print(batch["attention_mask"].shape)
print(batch["labels"].shape)

torch.Size([32, 50])
torch.Size([32, 50])
torch.Size([32])


In [44]:
input_ids = batch["input_ids"]

output = encoder(input_ids)
print(output.shape)

torch.Size([32, 50, 32])


# Modularize

## Data

In [ ]:
class IMDBDataset(Dataset):

    '''
        input_ids: Indices of the input sequence according to a selected tokenizer's corpus.
        attention_mask: Tells which tokens are real text and which tokens are padding.
        labels: Sentiment of the sequence. Binary value - 0 (negative) and 1 (positive).
    '''

    def __init__(self, split="train", max_length=128):
        dataset = load_dataset("imdb", split=split)

        tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

        encodings = tokenizer(
            list(dataset["text"]),
            truncation=True,
            stride=32,
            return_overflowing_tokens=True,
            padding="max_length",
            max_length=max_length
        )

        # map chunks back to original samples
        sample_map = encodings["overflow_to_sample_mapping"]

        self.input_ids = encodings["input_ids"]
        self.attention_mask = encodings["attention_mask"]
        labels = dataset["label"]
        self.labels = [labels[i] for i in sample_map]

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            "input_ids": torch.tensor(self.input_ids[idx]),
            "attention_mask": torch.tensor(self.attention_mask[idx]),
            "labels": torch.tensor(self.labels[idx])
        }

In [39]:
class ActionSequenceTokenizer:

    def __init__(self, max_length=5, stride=0, pad_token=0):
        self.max_length = max_length
        self.stride = stride
        self.pad_token = pad_token


    def encode(self, seq):

        seq = list(seq)

        sequences = []
        masks = []

        start = 0

        while start < len(seq):

            chunk = seq[start:start + self.max_length]

            mask = [1] * len(chunk)

            # padding if needed
            if len(chunk) < self.max_length:
                pad_size = self.max_length - len(chunk)
                chunk = chunk + [self.pad_token] * pad_size
                mask = mask + [0] * pad_size

            sequences.append(chunk)
            masks.append(mask)

            if len(seq) <= self.max_length:
                break

            start += self.max_length - self.stride

        return sequences, masks


class MLSmallDataset(Dataset):

    '''
        input_ids: Indices of the input sequence according to a selected tokenizer's corpus.
        labels: Score (rating) of the sequence.
    '''

    def __init__(self, csv_path, max_length=50, stride=0):

        ratings_df = pd.read_csv(csv_path)

        item_ids = ratings_df["movieId"].unique()
        item2idx = {item: idx+1 for idx, item in enumerate(item_ids)}
        ratings_df["movieId"] = ratings_df["movieId"].map(item2idx)

        item_sequence = (
            ratings_df
            .sort_values(['userId','timestamp'])
            .groupby('userId')['movieId']
            .agg(list)
        )

        sentences = item_sequence.tolist()
        

        tokenizer = ActionSequenceTokenizer(max_length, stride)

        self.input_ids = []
        self.attention_mask = []
        self.labels = []

        for seq in sentences:

            seq_input = seq[:-1]
            label = seq[-1]

            ids_list, masks_list = tokenizer.encode(seq_input)

            for ids, mask in zip(ids_list, masks_list):
                self.input_ids.append(ids)
                self.attention_mask.append(mask)
                self.labels.append(label)

        self.num_items = len(item2idx) + 1
        self.input_ids = torch.tensor(self.input_ids)
        self.attention_mask = torch.tensor(self.attention_mask)
        self.labels = torch.tensor(self.labels)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):

        return {
            "input_ids": self.input_ids[idx],
            "attention_mask": self.attention_mask[idx],
            "labels": self.labels[idx]
        }

# Init

## Data

In [40]:
ml_dataset = MLSmallDataset(
    csv_path=MOVIES_SMALL_DIR / "ratings.csv",
    max_length=50,
    stride=5
)

In [41]:
batch_size = 32

loader = DataLoader(
    ml_dataset,
    batch_size=batch_size,
    shuffle=True
)

# LACLRec